# Análise de Emoções (GoEmotions) — Treino no Colab

Treino pesado de **BERT** e **RoBERTa** em GPU do Colab. Os scripts são
os mesmos do repositório local — aqui apenas instalamos as dependências e
executamos em uma GPU mais rápida.

**Antes de começar:** menu `Ambiente de execução` → `Alterar o tipo de ambiente`
→ selecione uma **GPU L4** (recomendada) ou **T4**. Os `configs/*.yaml` já estão
ajustados para a L4 (24 GB).

## 1. Verificar a GPU

In [1]:
%pip install -q torch

import sys, platform, importlib.util
import torch

# Ambiente de execução
if importlib.util.find_spec("google.colab") is not None:
    ambiente = "Google Colab"
else:
    ambiente = "Local / VSCode"
# GPU
try:
    if torch.cuda.is_available():
        nome_gpu  = torch.cuda.get_device_name(0)
        vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
        gpu_info  = f"{nome_gpu}  ({vram_gb:.1f} GB VRAM)"
    else:
        gpu_info  = "Nenhuma GPU detectada"
except ImportError:
    gpu_info = "torch não instalado ainda"

print("=" * 50)
print(f"  Ambiente : {ambiente}")
print(f"  Python   : {sys.version.split()[0]}")
print(f"  SO       : {platform.system()} {platform.release()}")
print(f"  GPU      : {gpu_info}")
print("=" * 50)

  Ambiente : Google Colab
  Python   : 3.12.13
  SO       : Linux 6.6.122+
  GPU      : NVIDIA L4  (23.7 GB VRAM)


## 2. Obter o código (repositório privado)

O repositório é privado, então o clone exige um **token de acesso**:

1. No GitHub: **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Em *Repository access*
   selecione apenas `Trabalho_Final_PLN`; em *Permissions* defina
   **Contents: Read-only**.
2. No Colab: clique no ícone de **chave 🔑** (barra lateral esquerda) e
   adicione um secret chamado **`GITHUB_TOKEN`** com o valor do token.
   Ative o *Notebook access*.

A célula abaixo lê o token dos *Secrets* (sem expô-lo) e clona o repositório.

In [2]:
from google.colab import userdata

# Garante que estamos no diretório raiz
%cd /content

GITHUB_USER = "Gustavo-Piccoli"
REPO = "Trabalho_Final_PLN"
DEST_DIR = "Trabalho_Final_PLN"  # <-- Mude aqui para o diretório desejado
token = userdata.get("TF_PLF")  # lido dos Secrets do Colab

# Exclui os repositórios baixados anteriormente para evitar duplicatas/aninhamentos
!rm -rf {REPO} {DEST_DIR}

!git clone https://{token}@github.com/{GITHUB_USER}/{REPO}.git {DEST_DIR}
%cd {DEST_DIR}


/content
Cloning into 'Trabalho_Final_PLN'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 49 (delta 12), reused 46 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 10.37 MiB | 10.41 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/Trabalho_Final_PLN


## 3. Instalar dependências

O PyTorch já vem pré-instalado no Colab, então instalamos apenas o restante.

In [3]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 139.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 144.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 134.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import yaml

def otimizar_config_l4(caminho_arquivo):
    with open(caminho_arquivo, 'r') as f:
        config = yaml.safe_load(f)

    # Garante que a seção de treinamento existe
    if 'training' not in config:
        config['training'] = {}

    # Aplica as otimizações para a GPU L4
    config['training']['per_device_train_batch_size'] = 64  # Pode tentar 128 se max_length for pequeno
    config['training']['per_device_eval_batch_size'] = 64
    config['training']['fp16'] = True  # Essencial para GPUs modernas como a L4
    config['training']['dataloader_num_workers'] = 2

    # Salva o arquivo atualizado
    with open(caminho_arquivo, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

    print(f"[OK] Arquivo {caminho_arquivo} otimizado para L4!")

# Otimiza os arquivos anexados
otimizar_config_l4('configs/bert.yaml')
otimizar_config_l4('configs/roberta.yaml')

## 4. Treinar BERT (baseline)

Os `configs/*.yaml` já estão ajustados para a GPU **L4** (batch 64). Se você
selecionar uma GPU com menos memória (ex.: **T4**, 16 GB) e ocorrer erro de
falta de memória (*out of memory*), reduza `per_device_train_batch_size` para
32 no YAML.

In [4]:
!python -m src.train --config configs/bert.yaml

2026-06-10 14:01:20.772824: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 14:01:20.840953: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[info] CUDA disponivel: True | dispositivo: NVIDIA L4
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 332kB/s]
config.json: 100% 570/570 [00:00<00:00, 3.66MB/s]
vocab.txt: 232kB [00:00, 7.00MB/s]
tokenizer.json: 466kB [00:00, 1.23MB/s]
README.md: 9.40kB [00:00, 38.5MB/s]
simplified/train-00000-of-00001.parquet: 100% 2.77M/2.77M [00:01<00:00, 2.34MB/s]
simplified/valid

## 5. Treinar RoBERTa (modelo principal)

In [5]:
!python -m src.train --config configs/roberta.yaml

2026-06-10 14:08:18.002506: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 14:08:18.070799: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[info] CUDA disponivel: True | dispositivo: NVIDIA L4
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 175kB/s]
config.json: 100% 481/481 [00:00<00:00, 3.79MB/s]
vocab.json: 899kB [00:00, 1.44MB/s]
merges.txt: 456kB [00:00, 1.62MB/s]
tokenizer.json: 1.36MB [00:00, 1.91MB/s]
Tokenizando e construindo rotulos multi-hot: 100% 43410/43410 [00:01<00:00, 31063.73 examples/s]

## 6. Avaliação detalhada (relatórios + gráficos)

In [6]:
!python -m src.evaluate --model_dir results/bert/best_model --output_dir results/bert
!python -m src.evaluate --model_dir results/roberta/best_model --output_dir results/roberta

[info] dispositivo: cuda
2026-06-10 14:14:09.398726: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-10 14:14:09.467425: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Tokenizando e construindo rotulos multi-hot: 100% 43410/43410 [00:01<00:00, 26408.85 examples/s]
Tokenizando e construindo rotulos multi-hot: 100% 5426/5426 [00:00<00:00, 30194.83 examples/s]
Tokenizando e construindo rotulos multi-hot: 100% 5427/5427 [00:00<00:00, 30841.52 examples/s]
You're using a BertTokenizerFast tokenizer. Please note that w

## 7. Baixar os resultados

In [ ]:
import zipfile
import os
from google.colab import files

EXCLUIR = {".safetensors", "optimizer.pt"}

with zipfile.ZipFile("results.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk("results"):
        for fname in filenames:
            if not any(fname == e or fname.endswith(e) for e in EXCLUIR):
                zf.write(os.path.join(root, fname))

files.download("results.zip")

In [14]:
from google.colab import drive
drive.mount('/content/Colab_PLN')

Mounted at /content/Colab_PLN


In [8]:
import os

file_path = 'results.zip'
if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"O tamanho do arquivo {file_path} é: {size_mb:.2f} MB")
else:
    print(f"O arquivo {file_path} não foi encontrado no diretório atual.")

O tamanho do arquivo results.zip é: 3072.19 MB


In [15]:
import shutil
import os

origem = 'results.zip'
destino_dir = '/content/drive/Colab_PLN'
destino = f'{destino_dir}/results.zip'

# Garante que o diretório de destino exista
os.makedirs(destino_dir, exist_ok=True)

if os.path.exists(origem):
    print("Copiando arquivo para o Google Drive...")
    try:
        shutil.copy2(origem, destino)
        print(f"Cópia concluída com sucesso!\nO arquivo está salvo no seu Drive como: {destino}")
        print("Acesse drive.google.com para fazer o download com mais estabilidade.")
    except Exception as e:
        print(f"Erro ao copiar o arquivo: {e}")
else:
    print("O arquivo results.zip não foi encontrado no diretório atual.")

Copiando arquivo para o Google Drive...
Cópia concluída com sucesso!
O arquivo está salvo no seu Drive como: /content/drive/Colab_PLN/results.zip
Acesse drive.google.com para fazer o download com mais estabilidade.
